# 3장 Looking Inside Large Language Models — SageMaker AI Studio(JupyterLab)용

<!-- origin-notice -->
> **원본 노트북:** [chapter03/Chapter 3 - Looking Inside LLMs.ipynb](https://github.com/HandsOnLLM/Hands-On-Large-Language-Models/blob/main/chapter03/Chapter%203%20-%20Looking%20Inside%20LLMs.ipynb)  
> **원본 저장소:** [HandsOnLLM/Hands-On-Large-Language-Models](https://github.com/HandsOnLLM/Hands-On-Large-Language-Models) (Apache License 2.0, © 2024 Jay Alammar & Maarten Grootendorst)  
> 이 노트북은 원본을 SageMaker AI Studio에서 실행되도록 수정하고 한국어 설명·보충 실험을 추가한 것입니다. 변경 내용은 노트북 안의 "책과 다를 수 있는 점"에 적혀 있습니다.

원본: HandsOnLLM/Hands-On-Large-Language-Models `chapter03` 노트북(Apache-2.0)을 수정·보충했습니다. 권장 인스턴스 `ml.g4dn.xlarge`(T4 16GB), 스토리지 30GB 이상. 1·2장과 같은 space에서 실행하면 Phi-3 가중치 캐시를 다시 씁니다.

| 절 | 하는 일 | 책 위치(PDF 쪽) |
|---|---|---|
| 1 | Phi-3를 불러와 생성해 보고, 모델 구조(`print(model)`)를 읽기 | 117–121 |
| 2 | 순전파(forward pass)의 세 부분: 토크나이저 → Transformer 블록 묶음 → LM head | 121–124 |
| 3 | 확률 분포에서 다음 토큰 하나 고르기(디코딩) (+ 상위 10개 확률과 temperature) | 124–127 |
| 4 | 토큰을 병렬로 처리한다는 것의 의미와 컨텍스트 크기 | 127–129 |
| 5 | KV 캐시로 생성 속도 높이기 (+ KV 캐시 메모리 계산, MHA vs GQA) | 129–132 |
| 6 | Transformer 블록 내부와 최신 개선점(RMSNorm, SwiGLU, GQA, RoPE)을 실제 모델에서 찾기 | 132–157 |

`(+ …)` 표시는 책에 없는 보충 실험입니다. 셀은 위에서부터 차례로 실행하세요.

## 0. 환경 점검

이 장은 Phi-3만 쓰므로 추가 패키지가 거의 필요 없습니다. `MODEL_ID`, `DEVICE`를 여기서 한 번 정해 두고 아래 셀들이 함께 씁니다(책 코드는 `"cuda"`를 직접 적었습니다).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!df -h ~ | tail -1
import importlib, subprocess, sys
missing = [p for m, p in {"accelerate": "accelerate"}.items() if importlib.util.find_spec(m) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
import torch, transformers, time, gc
import pandas as pd
print("torch", torch.__version__, "| CUDA", torch.cuda.is_available(), "| transformers", transformers.__version__)

MODEL_ID = "microsoft/Phi-3-mini-4k-instruct"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

## 1. 학습된 Transformer LLM의 입력과 출력 (PDF 117–121)

1장과 같은 방식으로 모델·토크나이저·pipeline을 만듭니다. 이번에는 `max_new_tokens=50`으로 짧게 생성합니다.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map=DEVICE,
    torch_dtype="auto",        # transformers 5.x에서는 dtype="auto" (기존 이름은 경고만 나고 동작)
    trust_remote_code=False,   # 책 본문은 True. 최신 transformers에는 Phi-3가 내장되어 False가 안전
)
generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    return_full_text=False,
    max_new_tokens=50,
    do_sample=False,
)

**관찰 포인트.** 이번 프롬프트는 1장과 달리 `messages` 리스트가 아니라 **문자열**입니다. 그래서 채팅 템플릿(`<|user|>` … `<|assistant|>`)이 붙지 않고, 모델은 질문에 답하기보다 문서를 이어 쓰는 쪽으로 동작합니다. 책 출력에 "Solution 1:" 같은 엉뚱한 머리말이 나오는 이유가 이것입니다.

In [ ]:
prompt = "Write an email apologizing to Sarah for the tragic gardening mishap. Explain how it happened."
output = generator(prompt)
print(output[0]['generated_text'])

모델 구조를 출력합니다. **읽는 법:**

- `embed_tokens`: `Embedding(32064, 3072)` — 어휘 32,064개마다 3,072차원 벡터 하나(2장의 토큰 임베딩).
- `layers`: `32 x Phi3DecoderLayer` — Transformer 블록 32개가 차례로 쌓여 있습니다(1장 Q&A의 "층 쌓기").
- `lm_head`: `Linear(3072 → 32064)` — 마지막 벡터를 어휘 전체에 대한 점수로 바꿉니다.

> **책과 다를 수 있는 점.** transformers 버전에 따라 출력 모양이 조금 다릅니다. 예를 들어 최신 버전에서는 `rotary_emb`가 각 층의 `self_attn` 안이 아니라 `Phi3Model` 바로 아래에 한 번만 나타날 수 있습니다. 숫자(32064, 3072, 9216, 16384, 8192)는 같습니다.

In [ ]:
print(model)

### (+ 보충) 설정값(config)에서 3장 개념 찾기

`print(model)`에 안 보이는 설계값은 `model.config`에 있습니다. 3장에서 나오는 개념과 연결해 봅니다.

In [ ]:
c = model.config
head_dim = c.hidden_size // c.num_attention_heads
n_params = sum(p.numel() for p in model.parameters())
pd.DataFrame([
    ("num_hidden_layers", c.num_hidden_layers, "Transformer 블록 수"),
    ("hidden_size", c.hidden_size, "토큰 벡터 차원"),
    ("num_attention_heads", c.num_attention_heads, "어텐션 헤드 수(쿼리)"),
    ("num_key_value_heads", c.num_key_value_heads, "키·값 헤드 수. 쿼리 헤드와 같으면 MHA, 적으면 GQA"),
    ("head_dim", head_dim, "헤드 하나의 차원 = hidden_size / heads"),
    ("intermediate_size", c.intermediate_size, "피드포워드 신경망 중간 차원"),
    ("max_position_embeddings", c.max_position_embeddings, "컨텍스트 길이(Phi-3-mini-4k는 4,096)"),
    ("sliding_window", getattr(c, "sliding_window", None), "로컬(슬라이딩 윈도) 어텐션 범위"),
    ("rope_theta", getattr(c, "rope_theta", None) or (getattr(c, "rope_parameters", None) or {}).get("rope_theta"), "RoPE 회전 주파수의 기준값"),
    ("vocab_size", c.vocab_size, "어휘 크기"),
    ("parameters", f"{n_params/1e9:.2f} B", "전체 파라미터 수"),
], columns=["config", "value", "의미"]).set_index("config")

## 2–3. 순전파의 구성 요소와 다음 토큰 고르기 (PDF 121–127)

`model.model`은 Transformer 블록 묶음(LM head 앞까지), `model.lm_head`는 마지막 선형층입니다. 둘을 따로 호출해서 "블록 묶음이 벡터를 만들고 → LM head가 어휘 점수로 바꾼다"는 흐름을 직접 확인합니다. 마지막 위치의 점수 중 가장 큰 것을 고르면(`argmax`) 탐욕적 디코딩(greedy decoding)입니다. temperature를 0으로 두는 것과 같습니다.

In [ ]:
prompt = "The capital of France is"
input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(DEVICE)

with torch.no_grad():
    model_output = model.model(input_ids)              # Transformer 블록 묶음의 출력
    lm_head_output = model.lm_head(model_output[0])   # 어휘 점수(logits)

In [ ]:
token_id = lm_head_output[0, -1].argmax(-1)
tokenizer.decode(token_id)

**관찰 포인트.** 모양은 `[배치, 토큰 수, 차원]`입니다. 토큰마다 3,072차원 벡터가 나오고(`model_output`), LM head를 지나면 토큰마다 32,064개 점수가 나옵니다(`lm_head_output`).

> **책과 다를 수 있는 점.** 책은 토큰 수를 6으로 보여 줍니다(`[1, 6, 3072]`). 현재 Hugging Face의 Phi-3 토크나이저는 문장 시작 토큰 `<s>`를 붙이지 않아 5가 나올 수 있습니다(2장 노트북 1절 참고).

In [ ]:
print(model_output[0].shape, lm_head_output.shape)

### (+ 보충) 상위 10개 후보와 temperature

`argmax`는 1등만 봅니다. 점수(logits)를 softmax로 확률로 바꾸면 2등 이하 후보도 보입니다. **temperature** T는 softmax 전에 점수를 T로 나누는 값입니다. T가 작으면 1등에 확률이 몰리고, 크면 고르게 퍼집니다. 6장에서 샘플링 파라미터를 다룰 때 다시 나옵니다.

In [ ]:
logits = lm_head_output[0, -1].float()
top = torch.topk(logits, 10).indices
table = {"token": [repr(tokenizer.decode(t)) for t in top]}
for T in [0.3, 1.0, 1.5]:
    probs = torch.softmax(logits / T, dim=-1)
    table[f"T={T}"] = [round(probs[t].item(), 3) for t in top]
pd.DataFrame(table)

T=1.0에서 실제로 10번 뽑아 봅니다(`torch.multinomial`). `do_sample=True`일 때 생성 결과가 매번 달라지는 이유입니다.

In [ ]:
torch.manual_seed(0)
probs = torch.softmax(logits, dim=-1)
[tokenizer.decode(i) for i in torch.multinomial(probs, num_samples=10, replacement=True)]

## 4. 토큰 병렬 처리와 컨텍스트 크기 (PDF 127–129)

입력 토큰은 각자의 "계산 흐름(stream)"을 타고 Transformer 블록을 **동시에** 지납니다(1장 Q&A의 "토큰 방향 병렬"). 흐름의 개수 상한이 컨텍스트 크기이고, Phi-3-mini-4k는 4,096입니다. 생성할 때 다음 토큰을 정하는 데 쓰는 것은 **마지막 흐름의 출력**뿐입니다.

아래 셀은 나머지 흐름도 버리지 않고 모두 디코딩해 봅니다. 위치마다 "그 위치까지 보고 예측한 다음 토큰"이 나옵니다. 학습할 때는 이 예측들을 정답과 한 번에 비교합니다. 1장 Q&A에서 말한 교사 강요(teacher forcing)와 병렬 학습이 이것입니다.

In [ ]:
ids = input_ids[0]
preds = lm_head_output[0].argmax(-1)
pd.DataFrame({
    "입력(여기까지 봄)": [repr(tokenizer.decode(ids[:i+1])) for i in range(len(ids))],
    "예측한 다음 토큰": [repr(tokenizer.decode(p)) for p in preds],
})

## 5. 키·값 캐싱으로 생성 속도 높이기 (PDF 129–132)

토큰 하나를 생성할 때마다 앞선 모든 토큰을 다시 계산하면 낭비입니다. 앞 토큰들의 어텐션 키(K)·값(V) 벡터는 바뀌지 않으므로 저장해 두고 재사용합니다. 이것이 **KV 캐시**입니다. `use_cache`로 켜고 끌 수 있습니다.

T4 기준으로 책에서는 캐시 사용 약 6.7초, 미사용 약 21.9초가 걸렸습니다. 원본 `%%timeit`은 7회 반복이라 미사용 셀이 2분 넘게 걸리므로, 여기서는 `-r 3`(3회)으로 줄였습니다.

In [ ]:
prompt = "Write a very long email apologizing to Sarah for the tragic gardening mishap. Explain how it happened."
input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(DEVICE)

In [ ]:
%%timeit -n 1 -r 3
generation_output = model.generate(input_ids=input_ids, max_new_tokens=100, use_cache=True)

In [ ]:
%%timeit -n 1 -r 3
generation_output = model.generate(input_ids=input_ids, max_new_tokens=100, use_cache=False)

### (+ 보충) 생성 길이에 따라 차이가 어떻게 벌어지나

캐시가 없으면 t번째 토큰을 만들 때 앞의 t개를 전부 다시 계산하므로, 생성이 길어질수록 차이가 커집니다. `min_new_tokens`로 길이를 고정하고 토큰당 시간을 잽니다(미사용 쪽은 1분 정도 걸립니다).

In [ ]:
def gen_time(n, use_cache):
    if DEVICE == "cuda": torch.cuda.synchronize()
    t = time.perf_counter()
    model.generate(input_ids=input_ids, max_new_tokens=n, min_new_tokens=n, do_sample=False, use_cache=use_cache)
    if DEVICE == "cuda": torch.cuda.synchronize()
    return time.perf_counter() - t

rows = []
for n in [25, 50, 100]:
    a, b = gen_time(n, True), gen_time(n, False)
    rows.append({"new_tokens": n, "cache_s": round(a, 2), "no_cache_s": round(b, 2),
                 "ms/token (cache)": round(a / n * 1000), "ms/token (no cache)": round(b / n * 1000),
                 "speedup": round(b / a, 1)})
pd.DataFrame(rows).set_index("new_tokens")

### (+ 보충) KV 캐시는 메모리를 얼마나 쓰나: MHA vs GQA

캐시는 공짜가 아닙니다. 토큰 하나당 `2(K와 V) × 층 수 × KV 헤드 수 × 헤드 차원 × 바이트 수`만큼 GPU 메모리를 씁니다. 먼저 실제 캐시 텐서를 재서 공식과 맞는지 확인합니다.

In [ ]:
def kv_bytes_per_token(cfg, bytes_per_value=2):
    hd = getattr(cfg, "head_dim", None) or cfg.hidden_size // cfg.num_attention_heads
    return 2 * cfg.num_hidden_layers * cfg.num_key_value_heads * hd * bytes_per_value

def cache_tensors(cache):            # transformers 버전마다 캐시 객체 구조가 다름
    if hasattr(cache, "layers"):     # 5.x
        return [t for l in cache.layers for t in (l.keys, l.values)]
    if hasattr(cache, "key_cache"):  # 4.x DynamicCache
        return list(cache.key_cache) + list(cache.value_cache)
    return [t for kv in cache for t in kv]

with torch.no_grad():
    out = model(input_ids, use_cache=True)
measured = sum(t.numel() * t.element_size() for t in cache_tensors(out.past_key_values))
n_tok = input_ids.shape[1]
print(f"토큰 {n_tok}개 → 실측 {measured/1024:.0f} KiB, 공식 {kv_bytes_per_token(model.config, model.dtype.itemsize)*n_tok/1024:.0f} KiB")
del out

이제 같은 공식으로 **4K 컨텍스트를 꽉 채웠을 때**를 계산하고, 3장 144쪽의 **GQA(grouped-query attention)**를 쓰는 모델과 비교합니다. 비교 대상 Qwen2.5-3B-Instruct는 설정 파일만 내려받습니다(가중치 안 받음). Phi-3-mini는 KV 헤드가 쿼리 헤드와 같은 32개(MHA)이고, Qwen2.5-3B는 쿼리 16개가 KV 2개를 나눠 씁니다(GQA).

In [ ]:
from transformers import AutoConfig
cfgs = {"Phi-3-mini-4k (MHA)": model.config,
        "Qwen2.5-3B (GQA)": AutoConfig.from_pretrained("Qwen/Qwen2.5-3B-Instruct")}
rows = []
for name, cfg in cfgs.items():
    per_tok = kv_bytes_per_token(cfg)            # bf16/fp16 = 2바이트
    rows.append({"model": name, "layers": cfg.num_hidden_layers,
                 "q_heads": cfg.num_attention_heads, "kv_heads": cfg.num_key_value_heads,
                 "KiB/token": round(per_tok / 1024, 1),
                 "GiB @4096 tokens": round(per_tok * 4096 / 1024**3, 2),
                 "GiB @4096 x batch 8": round(per_tok * 4096 * 8 / 1024**3, 1)})
pd.DataFrame(rows).set_index("model")

**읽는 법.** Phi-3-mini는 컨텍스트 4K 한 건에 KV 캐시만 약 1.5GiB, 동시 요청 8건이면 12GiB입니다. T4(16GB)에서는 가중치(약 7.6GB)와 합치면 넘칩니다. GQA는 K·V 헤드를 여러 쿼리 헤드가 공유해 캐시를 크게 줄입니다. 최근 오픈 모델 대부분이 GQA를 쓰는 이유입니다.

## 6. Transformer 블록 내부와 최근 개선점 (PDF 132–157)

책 6절의 개념을 `print(model)` 출력의 실제 모듈 이름과 짝지어 봅니다.

| 책의 개념 | Phi-3에서 찾을 곳 | 확인할 숫자 |
|---|---|---|
| 셀프 어텐션의 Q·K·V 투영 | `self_attn.qkv_proj` | 출력 9216 = 3 × 3072 (Q·K·V를 한 번에) → KV 헤드도 32개, 즉 GQA 아님 |
| 어텐션 출력 합치기 | `self_attn.o_proj` | 3072 → 3072 |
| 피드포워드 신경망 + SwiGLU | `mlp.gate_up_proj`, `mlp.down_proj`, `activation_fn: SiLU` | 16384 = 2 × 8192 (gate와 up을 한 번에) |
| 사전 정규화(pre-norm) + RMSNorm | `input_layernorm`, `post_attention_layernorm` (Phi3RMSNorm) | 어텐션·FFN **앞**에 정규화 |
| 회전 위치 임베딩(RoPE) | `rotary_emb` | 어텐션 단계에서 Q·K에 적용(입력에 더하지 않음) |
| 로컬/슬라이딩 윈도 어텐션 | `config.sliding_window` | 2047 |

아래 셀은 첫 번째 블록만 출력하고, 잔차 연결(residual connection)의 크기를 잽니다. 블록 출력 = 입력 x + 변화량 f(x)이므로, 층마다 |x|와 |f(x)|를 비교할 수 있습니다. 대부분의 층에서 |f(x)|가 |x|보다 작으면 블록이 벡터를 새로 만들기보다 **조금씩 고쳐 쓴다**는 뜻입니다(1장 Q&A의 x + f(x)). 첫 층(임베딩 직후 벡터가 작음)과 마지막 몇 층(다음 토큰 예측 쪽으로 크게 바꿈)은 예외가 흔합니다. `hidden_states`의 마지막 값은 최종 정규화(`norm`)가 적용된 값이라 비교에서 뺐습니다.

In [ ]:
print(model.model.layers[0])

with torch.no_grad():
    hs = model(input_ids, output_hidden_states=True).hidden_states   # 33개: 임베딩 + 32층
norms = pd.DataFrame({
    "layer": range(1, len(hs) - 1),
    "|x| (block input)": [hs[i-1][0].float().norm(dim=-1).mean().item() for i in range(1, len(hs) - 1)],
    "|f(x)| (change)": [(hs[i] - hs[i-1])[0].float().norm(dim=-1).mean().item() for i in range(1, len(hs) - 1)],
}).set_index("layer").round(1)
norms.iloc[::4]

In [ ]:
ax = norms.plot(logy=True, figsize=(7, 3.5), title="residual stream: block input vs. block update")
ax.set_xlabel("layer"); ax.set_ylabel("mean L2 norm (log)"); ax.grid(alpha=.3)

## 7. 정리

| 개념 | 이 노트북에서 본 것 |
|---|---|
| 순전파 = 토크나이저 → 블록 묶음 → LM head | `model.model`, `model.lm_head`를 따로 호출(2절) |
| 디코딩 | `argmax`(탐욕적), softmax + temperature + 샘플링(3절) |
| 병렬 처리 | 위치마다 다음 토큰 예측이 한 번에 나옴(4절) |
| KV 캐시 | 속도 차이 실측, 메모리 공식, MHA vs GQA(5절) |
| 블록 내부 | qkv_proj·SwiGLU·RMSNorm·RoPE, 잔차 연결(6절) |

**자율주행·Agent로 이어지는 점(책 밖).** 주행 VLA 모델이 행동(궤적) 토큰을 자기회귀로 생성하면 5절의 ms/token이 곧 제어 주기 예산이 됩니다. 카메라 여러 대·여러 프레임의 비전 토큰은 컨텍스트를 빠르게 채워 KV 캐시 메모리를 키웁니다. 그래서 차량용 모델은 GQA, 슬라이딩 윈도 어텐션, 토큰 수 줄이기 같은 기법을 함께 씁니다. Agent는 도구 호출 결과를 계속 컨텍스트에 붙이므로 대화가 길어질수록 같은 문제가 생기고, 이것이 프롬프트 캐싱 같은 서빙 기법이 중요한 이유입니다.

**끝나면** Studio에서 space를 **Stop** 하세요. EBS 볼륨(모델 캐시)은 남아 다음 장에서 재사용됩니다.